<span STYLE="font-size:150%"> 
    Segment microCT scans
</span>

Docker image: gnasello/slicer-env:2023-07-06 \
Latest update: 10 March 2023

- load image stack in Slicer
- segment mineralized tissue
- compute segmented statistics (volumes)

# Install libraries

Run the command below one time only, then comment it

In [ ]:
# !git clone https://github.com/gabnasello/pyslicer.git
# !PythonSlicer -m pip install -e pyslicer/
# !PythonSlicer -m pip install matplotlib

# import os
# os._exit(00)

# Load libraries

In [ ]:
import pyslicer as ps
import slicer
from pathlib import Path
import pandas as pd

# Volume input

## Load `.nrrd` file into 3D Slicer

Write the path of the `.nrrd` file and load it to Slicer

In [ ]:
# this cell is tagged 'parameters'
volume_file = 'microCT_volume/microCT_volume_preview.nrrd'
output_dir_path = 'microCT_calibration'
segment_file = 'spheres_calibration.seg.nrrd'
calibration_file = 'calibration_stats.csv'

In [ ]:
path = Path(volume_file)

# Remove image numbering _0000, _0001 ...
filename_output = path.stem[:-4]

In [ ]:
masterVolumeNode = slicer.util.loadNodeFromFile(volume_file)

Print spacing

In [ ]:
## mm
masterVolumeNode.GetSpacing()

Make ```microCT_calibration``` folder

In [ ]:
output_directory = Path(output_dir_path)

output_directory.mkdir(parents=True, exist_ok=True)

## Create segmentation-related nodes

Create segmentation node

In [ ]:
file_path = output_directory / segment_file

# Check if the file exists
if file_path.exists():
    segmentationNode = slicer.util.loadSegmentation(file_path.resolve(), properties={'name':"Segmentation"})
else:
    segmentationNode = ps.segmentation.segmentationNode(name='Segmentation')

Create temporary segment editor to get access to effects

In [ ]:
segmentEditorWidget, segmentEditorNode = ps.segmentation.segmentEditorWidget(segmentationNode = segmentationNode, 
                                                                             masterVolumeNode = masterVolumeNode)

# Segment calibration spheres

## Spheres 1 and 2 (highest densities)

### Otsu thresholding

Get automatic thresholding values, as indicated in [this script](https://github.com/jzeyl/3D-Slicer-Scripts/blob/db51967cc642837e8bae0fea1585a95074d8420b/load_dicom_modified_otsu.py#L56)

In [ ]:
methods = [
            'HUANG',
            'INTERMODES',
            'ISO_DATA',
            'KITTLER_ILLINGWORTH',
            'LI',
            'MAXIMUM_ENTROPY',
            'MOMENTS',
            'OTSU',
            'RENYI_ENTROPY',
            'SHANBHAG',
            'TRIANGLE',
            'YEN'
            ]

thresholds = dict.fromkeys(methods, None)
thresholds

Otsu thresholding

In [ ]:
method = 'OTSU'

threshold = ps.segmentation.compute_threshold(method = method, volumeNode = masterVolumeNode)

thresholds[method.upper()] = threshold

print(method + " threshold: " + str(threshold))
ps.volume.plot_histogram(masterVolumeNode, threshold = threshold, title = method, yscale='log')

### Create segments by thresholding

In [ ]:
segment_name = "Threshold"

segments_greyvalues = {
    segment_name: [thresholds['OTSU'], 65535.00],
    }

segments_greyvalues

In [ ]:
# Avoid overwrite of overlapping segments
segmentEditorNode.SetOverwriteMode(slicer.vtkMRMLSegmentEditorNode.OverwriteNone)

ps.segmentation.segments_by_thresholding(segments_greyvalues, 
                                         segmentationNode,
                                         segmentEditorNode,
                                         segmentEditorWidget)

### Remove Islands 

Remove smaller islands than 50.000 voxels

In [ ]:
minimum_size = 50000

ps.segmentation.remove_small_islands(minimum_size, segment_name, segmentEditorNode, segmentEditorWidget)

### Margins - shrink

Shrink 5×5×5 pixels (0.1 mm)

In [ ]:
shrink_pixels=-5

ps.segmentation.margin_segmentation(segmentationNode,
                                    masterVolumeNode,
                                    segmentEditorNode,
                                    segmentEditorWidget,
                                    segment_name=segment_name,
                                    margin_pixels=shrink_pixels,
                                    )

### Split islands to segments

min size 30.000 voxels

In [ ]:
minimum_size = 30000

ps.segmentation.split_islands(minimum_size, segment_name, segmentEditorNode, segmentEditorWidget)

### Margins - grow

Grow same pixels that were first shrunk

In [ ]:
grow_pixels = -shrink_pixels 

ps.segmentation.margin_segmentation(
                                    segmentationNode,
                                    masterVolumeNode,
                                    segmentEditorNode,
                                    segmentEditorWidget,
                                    margin_pixels=grow_pixels,
                                    apply_to_all_visible=True
                                )

### Select sphere segments (roundness)

Identify sphere-like segments (roundness / flatness / elongation)

In [ ]:
extra_keys = ["centroid_ras", "feret_diameter_mm", "surface_area_mm2", "roundness", 
              "flatness", "elongation","principal_moments"]

stats = ps.segmentation.segment_statistics(segmentationNode, masterVolumeNode, extra_keys)

In [ ]:
# 3) Get a compact wide layout table for quick viewing or export
df_stats = ps.segmentation.stats_to_dataframe(stats, orientation="wide")
df_stats.head()

In [ ]:
# convert Roundness to numeric (coerce errors -> NaN)
df_stats["Roundness"] = pd.to_numeric(df_stats["Roundness"], errors="coerce")

# compute distance from 1
df_stats["roundness_distance"] = (df_stats["Roundness"] - 1).abs()

# get the two closest segments
df_spheres = df_stats.nsmallest(2, "roundness_distance")

df_spheres['segment']

In [ ]:
# List of segment names you want to KEEP
keep_segments = df_spheres['segment'].to_list()

ps.segmentation.keep_segments_by_name(keep_segments, segmentationNode)

## Spheres 3 (manual selection)

<span style="color:red; font-size:200%">Manual Task</span>

### Flood filling segmentation by clicking on the calibration sphere

In [ ]:
seg_name = "Sphere"

# Add a segment to the segmentation node
segmentationNode.GetSegmentation().AddEmptySegment("Sphere")

# Activate the Flood Filling effect
segmentEditorNode.SetSelectedSegmentID(seg_name)

# ------------------------------------------------------------------
# Activate Flood Filling effect
# ------------------------------------------------------------------
segmentEditorWidget.setActiveEffectByName("Flood filling")
effect = segmentEditorWidget.activeEffect()

# ------------------------------------------------------------------
# Set Flood Filling parameters
# ------------------------------------------------------------------
effect.setParameter("IntensityTolerance", "1000")
effect.setParameter("NeighborhoodSizeMm", "2.5")

In [ ]:

print("Stopping here for manual intervention...")
raise SystemExit


<span style="color:red; font-size:200%">Manual Task</span>

Click on the region of interest

### Margins

Grow 5×5×5 pixels (0.1 mm)

In [ ]:
grow_pixels=5

ps.segmentation.margin_segmentation(segmentationNode,
                                    masterVolumeNode,
                                    segmentEditorNode,
                                    segmentEditorWidget,
                                    segment_name=seg_name,
                                    margin_pixels=grow_pixels,
                                    )

Shrink 5×5×5 pixels (0.1 mm)

In [ ]:
shrink_pixels = -grow_pixels

ps.segmentation.margin_segmentation(segmentationNode,
                                    masterVolumeNode,
                                    segmentEditorNode,
                                    segmentEditorWidget,
                                    segment_name=seg_name,
                                    margin_pixels=shrink_pixels,
                                    )

### OPTIONAL - Enclosing sphere segmentation created

In [ ]:
import slicer
import vtk
import numpy as np
import random

# =========================
# USER INPUT
# =========================
SOURCE_SEGMENTATION_NAME = "Segmentation"
NEW_SEGMENTATION_NAME = "EnclosingSpheres"

# =========================
# MINIMUM ENCLOSING SPHERE
# =========================

def sphere_from(points):
    if len(points) == 0:
        return np.zeros(3), 0
    elif len(points) == 1:
        return points[0], 0
    elif len(points) == 2:
        center = (points[0] + points[1]) / 2
        radius = np.linalg.norm(points[0] - center)
        return center, radius
    elif len(points) == 3:
        A = points[1] - points[0]
        B = points[2] - points[0]
        cross = np.cross(A, B)
        denom = 2 * np.dot(cross, cross)
        if denom == 0:
            return sphere_from(points[:2])
        center = points[0] + (
            np.cross(cross, A) * np.dot(B, B) +
            np.cross(B, cross) * np.dot(A, A)
        ) / denom
        radius = np.linalg.norm(center - points[0])
        return center, radius
    else:
        A, B, C, D = points
        a = B - A
        b = C - A
        c = D - A
        M = np.vstack([a, b, c]).T
        if abs(np.linalg.det(M)) < 1e-6:
            return sphere_from(points[:3])
        center = np.linalg.solve(
            2 * M,
            np.array([np.dot(a, a), np.dot(b, b), np.dot(c, c)])
        ) + A
        radius = np.linalg.norm(center - A)
        return center, radius

def is_in_sphere(p, center, radius, eps=1e-6):
    return np.linalg.norm(p - center) <= radius + eps

def minimum_enclosing_sphere(points):
    P = points.copy()
    random.shuffle(P)

    center = np.zeros(3)
    radius = 0

    for i in range(len(P)):
        if not is_in_sphere(P[i], center, radius):
            center = P[i]
            radius = 0
            for j in range(i):
                if not is_in_sphere(P[j], center, radius):
                    center = (P[i] + P[j]) / 2
                    radius = np.linalg.norm(P[j] - center)
                    for k in range(j):
                        if not is_in_sphere(P[k], center, radius):
                            center, radius = sphere_from([P[i], P[j], P[k]])
                            for l in range(k):
                                if not is_in_sphere(P[l], center, radius):
                                    center, radius = sphere_from([P[i], P[j], P[k], P[l]])
    return center, radius

# =========================
# SLICER HELPERS
# =========================

def get_segment_surface_points(segmentationNode, segmentId):
    polyData = vtk.vtkPolyData()
    segmentationNode.GetClosedSurfaceRepresentation(segmentId, polyData)

    pointsVTK = polyData.GetPoints()
    if not pointsVTK:
        return None

    return np.array([
        pointsVTK.GetPoint(i)
        for i in range(pointsVTK.GetNumberOfPoints())
    ])

def add_sphere_segment(segmentationNode, name, color, center, radius):
    sphere = vtk.vtkSphereSource()
    sphere.SetCenter(center)
    sphere.SetRadius(radius)
    sphere.SetThetaResolution(64)
    sphere.SetPhiResolution(64)
    sphere.Update()

    segment = slicer.vtkSegment()
    segment.SetName(name)
    segment.SetColor(color)

    segment.AddRepresentation(
        slicer.vtkSegmentationConverter.GetSegmentationClosedSurfaceRepresentationName(),
        sphere.GetOutput()
    )

    segmentationNode.GetSegmentation().AddSegment(segment)

In [ ]:
# =========================
# MAIN PIPELINE
# =========================

sourceSeg = slicer.util.getNode(SOURCE_SEGMENTATION_NAME)

sphereSeg = slicer.mrmlScene.AddNewNodeByClass(
    "vtkMRMLSegmentationNode",
    NEW_SEGMENTATION_NAME
)
sphereSeg.CreateDefaultDisplayNodes()

segmentIds = vtk.vtkStringArray()
sourceSeg.GetSegmentation().GetSegmentIDs(segmentIds)

for i in range(segmentIds.GetNumberOfValues()):
    segId = segmentIds.GetValue(i)
    segment = sourceSeg.GetSegmentation().GetSegment(segId)

    name = segment.GetName()
    color = segment.GetColor()

    points = get_segment_surface_points(sourceSeg, segId)
    if points is None or len(points) == 0:
        print(f"Skipping empty segment: {name}")
        continue

    # Optional speed-up for very dense meshes
    if len(points) > 5000:
        points = points[np.random.choice(len(points), 5000, replace=False)]

    center, radius = minimum_enclosing_sphere(points)

    add_sphere_segment(
        sphereSeg,
        name,
        color,
        center,
        radius
    )

print("✅ Enclosing sphere segmentation created:", NEW_SEGMENTATION_NAME)

# Get sphere statistics

## Segment Statistics

Identify sphere-like segments (roundness / flatness / elongation)

In [ ]:
extra_keys = ["centroid_ras", "feret_diameter_mm", "surface_area_mm2", "roundness", 
              "flatness", "elongation","principal_moments"]

stats = ps.segmentation.segment_statistics(segmentationNode, masterVolumeNode, extra_keys)

In [ ]:
# 3) Get a compact wide layout table for quick viewing or export
df_stats = ps.segmentation.stats_to_dataframe(stats, orientation="wide")
df_stats.head()

## Material DataFrame

In [ ]:
sphere_names = ['Sphere_1', 'Sphere_2', 'Sphere_3']  
calibration_materials = ['polyacetol', 'borosilicate', 'ceramic']
material_densities = [1.4, 2.23, 3.85] # g/cm3

In [ ]:
# Create DataFrame
df_materials = pd.DataFrame({
    'Sphere_name': sphere_names,
    'Material': calibration_materials,
    'Density_g_cm3': material_densities
})

df_materials

In [ ]:
#import pandas as pd

# 1) Sort df_stats by Median ascending
df_stats_sorted = df_stats.sort_values('Median', ascending=True).reset_index(drop=True)

# 2) Create a rank index to align rows one to one
df_stats_sorted['order'] = range(len(df_stats_sorted))

df_materials['order'] = range(len(df_materials))

# 4) Merge by the synthetic order key
df_merged = (
    df_materials
    .merge(df_stats_sorted, on='order', how='left')
    .drop(columns='order')
)

df_merged.to_csv(output_directory/calibration_file, index=False)

df_merged
# df_merged now has df_stats columns plus a Material colum

# Rename Segments

In [ ]:
# 1) Get your Segmentation
seg = segmentationNode.GetSegmentation()

# 2) Build dictionary: current segment name -> Sphere_name
# df_merged must already exist in your Python environment
name_map = dict(zip(df_merged['segment'], df_merged['Sphere_name']))

# 3) Apply renaming inside Slicer
for i in range(seg.GetNumberOfSegments()):
    segmentId = seg.GetNthSegmentID(i)
    segment = seg.GetSegment(segmentId)
    old_name = segment.GetName()

    if old_name in name_map:
        new_name = name_map[old_name]
        print(f"Renaming '{old_name}' → '{new_name}'")
        segment.SetName(new_name)

# Export segments

## As seg.nrrd file (labelmap node)

In [ ]:
slicer.util.exportNode(segmentationNode, output_directory / segment_file)